In [2]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import pandas as pd

# Read the Facebook data
file_path = '/content/drive/MyDrive/dataset_Facebook.csv'
df = pd.read_csv(file_path, sep=';')

# Show the dataset size and the first 5 rows
print("Dataset shape:", df.shape)
display(df.head())

Dataset shape: (500, 19)


,Page total likes,Type,Category,Post Month,Post Weekday,Post Hour,Paid,Lifetime Post Total Reach,Lifetime Post Total Impressions,Lifetime Engaged Users,Lifetime Post Consumers,Lifetime Post Consumptions,Lifetime Post Impressions by people who have liked your Page,Lifetime Post reach by people who like your Page,Lifetime People who have liked your Page and engaged with your post,comment,like,share,Total Interactions
0,139441,Photo,2,12,4,3,0.0,2752,5091,178,109,159,3078,1640,119,4,79.0,17.0,100
1,139441,Status,2,12,3,10,0.0,10460,19057,1457,1361,1674,11710,6112,1108,5,130.0,29.0,164
2,139441,Photo,3,12,3,3,0.0,2413,4373,177,113,154,2812,1503,132,0,66.0,14.0,80
3,139441,Photo,2,12,2,10,1.0,50128,87991,2211,790,1119,61027,32048,1386,58,1572.0,147.0,1777
4,139441,Photo,2,12,2,3,0.0,7244,13594,671,410,580,6228,3200,396,19,325.0,49.0,393


In [5]:
# Check memory usage before optimization
memory_before = df.memory_usage(deep=True)
print(memory_before)

# Show the total in bytes and KB
total_before = memory_before.sum()
print("Total memory in bytes:", total_before)
print(f"Total memory in KB: {total_before / 1024:.2f}")

Index                                                                    132
Page total likes                                                        4000
Type                                                                   27023
Category                                                                4000
Post Month                                                              4000
Post Weekday                                                            4000
Post Hour                                                               4000
Paid                                                                    4000
Lifetime Post Total Reach                                               4000
Lifetime Post Total Impressions                                         4000
Lifetime Engaged Users                                                  4000
Lifetime Post Consumers                                                 4000
Lifetime Post Consumptions                                              4000

In [6]:
import numpy as np

def optimize_dtypes(df):
    optimized = df.copy()

    # Find the smallest integer type that fits each column
    for column in optimized.select_dtypes(include="integer").columns:
        smallest = optimized[column].min()
        largest = optimized[column].max()

        for dtype in [np.int8, np.int16, np.int32, np.int64]:
            limits = np.iinfo(dtype)
            if limits.min <= smallest and largest <= limits.max:
                optimized[column] = optimized[column].astype(dtype)
                break

    # Convert decimal numbers to float32
    for column in optimized.select_dtypes(include="float").columns:
        optimized[column] = optimized[column].astype("float32")

    # Store repeated post types as categories
    optimized["Type"] = optimized["Type"].astype("category")

    return optimized

In [7]:
# Apply the function and check memory usage
optimized_df = optimize_dtypes(df)
memory_after = optimized_df.memory_usage(deep=True)
print(memory_after)

# Calculate the percentage of memory saved
total_after = memory_after.sum()
percent_saved = (total_before - total_after) / total_before * 100

print(f"Memory before: {total_before:,} bytes")
print(f"Memory after: {total_after:,} bytes ({total_after / 1024:.2f} KB)")
print(f"Memory saved: {percent_saved:.2f}%")

Index                                                                   132
Page total likes                                                       2000
Type                                                                    888
Category                                                                500
Post Month                                                              500
Post Weekday                                                            500
Post Hour                                                               500
Paid                                                                   2000
Lifetime Post Total Reach                                              2000
Lifetime Post Total Impressions                                        2000
Lifetime Engaged Users                                                 1000
Lifetime Post Consumers                                                1000
Lifetime Post Consumptions                                             1000
Lifetime Pos

In [8]:
# Compare column types and memory savings
comparison = pd.DataFrame({
    "Before": df.dtypes,
    "After": optimized_df.dtypes,
    "Bytes saved": memory_before[df.columns] - memory_after[df.columns]
})

display(comparison.sort_values("Bytes saved", ascending=False))

,Before,After,Bytes saved
Type,object,category,26135
Category,int64,int8,3500
Post Month,int64,int8,3500
Post Hour,int64,int8,3500
Post Weekday,int64,int8,3500
comment,int64,int16,3000
Lifetime Post Consumers,int64,int16,3000
Lifetime Engaged Users,int64,int16,3000
Total Interactions,int64,int16,3000
Lifetime People who have liked your Page and engaged with your post,int64,int16,3000


In [9]:
from scipy.sparse import csr_matrix

# Convert the three columns to text
columns = ["Type", "Post Month", "Post Weekday"]
post_data = df[columns].astype(str)

# Create a table of zeros and ones
encoded_posts = pd.get_dummies(post_data, dtype=int)

# Convert the table to a sparse matrix
sparse_posts = csr_matrix(encoded_posts.to_numpy())

print("Matrix shape:", sparse_posts.shape)
print("Stored elements:", sparse_posts.nnz)

Matrix shape: (500, 23)
Stored elements: 1500


In [10]:
# Convert the sparse matrix to a dense array
dense_posts = sparse_posts.toarray()

# Show the first 5 rows
print(dense_posts[:5])

# Check that all values match the original table
print("All values match:", np.array_equal(dense_posts, encoded_posts.to_numpy()))

[[0 1 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0]
 [0 0 1 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0]
 [0 1 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0]
 [0 1 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0]
 [0 1 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0]]
All values match: True


In [11]:
# Convert the dense array back to CSR
restored_posts = csr_matrix(dense_posts)

# Compare the number of stored elements
print("Stored elements:", restored_posts.nnz)
print("Same count as 2(a):", restored_posts.nnz == sparse_posts.nnz)

Stored elements: 1500
Same count as 2(a): True


In [12]:
# Calculate the percentage of zeros
total_cells = dense_posts.size
zero_cells = total_cells - sparse_posts.nnz
sparsity = zero_cells / total_cells * 100

# Compare memory usage in bytes
dense_bytes = dense_posts.nbytes
sparse_bytes = (
    sparse_posts.data.nbytes
    + sparse_posts.indices.nbytes
    + sparse_posts.indptr.nbytes
)

print(f"Sparsity: {sparsity:.1f}%")
print(f"Dense memory: {dense_bytes:,} bytes")
print(f"Sparse memory: {sparse_bytes:,} bytes")

Sparsity: 87.0%
Dense memory: 92,000 bytes
Sparse memory: 20,004 bytes


In [13]:
import time

# Repeat the data 1,000 times
big_df = pd.concat([df] * 1000, ignore_index=True)
print("Dataset shape:", big_df.shape)

# Save each file and measure its write time
start = time.time()
big_df.to_csv("facebook_posts.csv", index=False)
csv_write = time.time() - start

start = time.time()
big_df.to_parquet("facebook_posts.parquet", index=False)
parquet_write = time.time() - start

start = time.time()
big_df.to_feather("facebook_posts.feather")
feather_write = time.time() - start

print(f"CSV write time: {csv_write:.3f} seconds")
print(f"Parquet write time: {parquet_write:.3f} seconds")
print(f"Feather write time: {feather_write:.3f} seconds")

Dataset shape: (500000, 19)
CSV write time: 6.347 seconds
Parquet write time: 0.396 seconds
Feather write time: 0.112 seconds


In [14]:
import os

# Measure how long each file takes to read
start = time.time()
pd.read_csv("facebook_posts.csv")
csv_read = time.time() - start

start = time.time()
pd.read_parquet("facebook_posts.parquet")
parquet_read = time.time() - start

start = time.time()
pd.read_feather("facebook_posts.feather")
feather_read = time.time() - start

# Get each file size in KB
csv_size = os.path.getsize("facebook_posts.csv") / 1024
parquet_size = os.path.getsize("facebook_posts.parquet") / 1024
feather_size = os.path.getsize("facebook_posts.feather") / 1024

# Show all results in one table
results = pd.DataFrame({
    "Format": ["CSV", "Parquet", "Feather"],
    "Size (KB)": [csv_size, parquet_size, feather_size],
    "Write time (s)": [csv_write, parquet_write, feather_write],
    "Read time (s)": [csv_read, parquet_read, feather_read]
})

display(results.round(3))

# Recommend the format that uses the least storage
smallest_format = results.loc[results["Size (KB)"].idxmin(), "Format"]
print(f"Recommendation: {smallest_format} because it uses the least storage for this dataset.")

,Format,Size (KB),Write time (s),Read time (s)
0,CSV,39023.846,6.347,4.415
1,Parquet,1247.360,0.396,0.288
2,Feather,4555.369,0.112,0.125


Recommendation: Parquet because it uses the least storage for this dataset.
